# Multi-Agent Verification System with QWEN-4 (Hugging Face)

This notebook tests the multi-agent verification system using the QWEN-4 model from Hugging Face instead of OpenRouter/Yandex.

## System Components:
- **Plaintiff Agent**: Generates initial comprehensive answer
- **Critic Agent**: Analyzes and critiques for logical errors
- **Librarian Agent**: Searches for sources and information
- **Jury Node**: Multiple independent evaluators
- **Judge Agent**: Makes final verdict
- **Final Agent**: Synthesizes final answer
- **Memory System**: Tracks history and improvements

## 1. Install Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'transformers',
    'torch',
    'langchain',
    'langgraph',
    'requests',
    'pydantic'
]

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

## 2. Imports and Setup

In [ ]:
# -*- coding: utf-8 -*-
"""Multi-Agent Verification System with QWEN-4 Model"""

import os
import json
import time
import requests
from typing import TypedDict, List, Dict, Any, Optional
from datetime import datetime

# LangChain imports
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END

# Transformers import for QWEN
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Pretty printing
class Colors:
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

print(f"{Colors.GREEN}✓ All imports successful{Colors.ENDC}")

## 3. Configuration and QWEN-4 LLM Setup

In [ ]:
# Configuration
CONFIG = {
    "model_name": "Qwen/Qwen2-4B-Instruct",  # QWEN-4B model
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "max_tokens": 512,
    "temperature": 0.6,
    "use_jury": True,
    "max_iterations": 2,
    "target_confidence": 0.85,
    "web_search": {
        "enabled": True,
        "max_results": 3,
    },
}

# Global logging
conversation_log = []

print(f"Device: {CONFIG['device']}")
print(f"Model: {CONFIG['model_name']}")
print(f"Max tokens per response: {CONFIG['max_tokens']}")

## 4. QWEN-4 LLM Provider

In [ ]:
class QWENLLM:
    """QWEN-4 LLM Provider using Hugging Face"""
    
    def __init__(self, model_name: str = "Qwen/Qwen2-4B-Instruct", device: str = "cpu"):
        """Initialize QWEN model"""
        print(f"Loading {model_name}...")
        
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto" if device == "cuda" else None,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            low_cpu_mem_usage=True
        )
        
        if device == "cpu":
            self.model = self.model.to(device)
        
        self.model.eval()
        print(f"✓ Model loaded on {device}")
    
    def __call__(self, prompt: str, max_tokens: int = 512, temperature: float = 0.6) -> str:
        """Generate text using QWEN"""
        try:
            # Tokenize input
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            
            # Generate output
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature,
                    top_p=0.9,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            
            # Decode output
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Extract only the generated part (remove prompt)
            if prompt in response:
                response = response[len(prompt):].strip()
            
            return response
        
        except Exception as e:
            print(f"❌ Error generating response: {e}")
            return f"Error: {str(e)}"

# Initialize QWEN LLM
print("\nInitializing QWEN-4 model...")
llm = QWENLLM(
    model_name=CONFIG["model_name"],
    device=CONFIG["device"]
)
print(f"{Colors.GREEN}✓ QWEN LLM ready{Colors.ENDC}")

## 5. Utility Functions

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimation (1 token ≈ 4 chars in Russian/English)"""
    return len(text) // 4 + len(text.split())

def print_header(text: str):
    """Print formatted header"""
    print(f"\n{Colors.BOLD}{Colors.HEADER}{'='*80}{Colors.ENDC}")
    print(f"{Colors.BOLD}{Colors.HEADER}{text.center(80)}{Colors.ENDC}")
    print(f"{Colors.BOLD}{Colors.HEADER}{'='*80}{Colors.ENDC}\n")

def print_agent_output(agent_name: str, output: str, iteration: int):
    """Print agent output with formatting"""
    tokens = estimate_tokens(output)
    print(f"\n{Colors.BOLD}{Colors.CYAN}[Iteration {iteration}] {agent_name.upper()}{Colors.ENDC}")
    print(f"{Colors.YELLOW}📊 Output tokens: {tokens}{Colors.ENDC}")
    print(f"{Colors.GREEN}{'-'*80}{Colors.ENDC}")
    print(output[:500] + ("..." if len(output) > 500 else ""))  # Limit output for notebook
    print(f"{Colors.GREEN}{'-'*80}{Colors.ENDC}")

def call_llm(prompt: str) -> str:
    """Call QWEN LLM"""
    response = llm(prompt, max_tokens=CONFIG["max_tokens"], temperature=CONFIG["temperature"])
    
    # Log conversation
    conversation_log.append({
        "timestamp": datetime.now().isoformat(),
        "prompt_tokens": estimate_tokens(prompt),
        "response_tokens": estimate_tokens(response),
    })
    
    return response

def web_search(query: str) -> str:
    """Simple web search simulation (or real search if API available)"""
    # For notebook, use mock search or real DuckDuckGo
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        params = {"q": query}
        response = requests.get("https://html.duckduckgo.com/", params=params, headers=headers, timeout=5)
        response.raise_for_status()
        
        import re
        snippets = re.findall(r'<a.*?href="([^"]*)".*?>([^<]*)</a>.*?<div[^>]*>([^<]*)<', response.text)
        
        results = []
        for i, (href, title, snippet) in enumerate(snippets[:3], 1):
            if href and title and snippet:
                results.append(f"{i}. {title.strip()}\n   {snippet.strip()[:150]}")
        
        return "\n\n".join(results) if results else "No search results found."
    except:
        # Fallback mock search
        return f"Search results for '{query}': [Mock search - use real API for production]"

print(f"{Colors.GREEN}✓ Utility functions loaded{Colors.ENDC}")

## 6. Agent State Definition

In [ ]:
class AgentState(TypedDict):
    """State for multi-agent system"""
    topic: str
    plaintiff_answer: str
    critic_answer: str
    librarian_sources: List[Dict[str, str]]
    jury_opinions: List[str]
    judge_verdict: str
    final_answer: str
    history: List[Dict[str, Any]]
    summary_memory: str
    iteration: int
    max_iterations: int
    stop: bool
    metrics: Dict[str, Any]
    rewards: List[float]
    generated_queries: List[str]

print(f"{Colors.GREEN}✓ Agent state defined{Colors.ENDC}")

## 7. Agent Functions

In [ ]:
def plaintiff_agent(state: AgentState) -> AgentState:
    """Generate initial answer - plaintiff agent"""
    prompt_template = PromptTemplate(
        input_variables=["topic", "summary_memory"],
        template="""[ROLE: PLAINTIFF - ИСТЕЦ]

ТЕМА РАССМОТРЕНИЯ:
{topic}

ПАМЯТЬ (предыдущие итерации):
{summary_memory}

Ты — экспертный аналитический агент, специализирующийся на создании обоснованных, проверяемых и логически связных ответов.

ТВОИ ОБЯЗАТЕЛЬСТВА:
1. Дай полный, структурированный ответ по теме.
2. Учитывай ПАМЯТЬ ПРОЦЕССА — ранее выявленные ошибки, замечания критиков и требования судьи.
3. Избегай голословных утверждений — подкрепляй каждый тезис аргументами.
4. Структурируй ответ: тезис → подробные аргументы → ограничения и оговорки → итоговый вывод.
5. Не повторяй ошибки, выявленные в прошлых итерациях.

ОТВЕТ (3-4 абзаца):
"""
    )
    
    prompt = prompt_template.format(
        topic=state['topic'],
        summary_memory=state['summary_memory'] or "Это первая итерация."
    )
    
    answer = call_llm(prompt)
    state["plaintiff_answer"] = answer
    print_agent_output("plaintiff", answer, state['iteration'])
    
    return state

print(f"{Colors.GREEN}✓ Plaintiff agent defined{Colors.ENDC}")

In [ ]:
def critic_agent(state: AgentState) -> AgentState:
    """Analyze and critique the answer"""
    prompt_template = PromptTemplate(
        input_variables=["plaintiff_answer", "summary_memory"],
        template="""[ROLE: CRITIC - КРИТИК]

АНАЛИЗИРУЕМЫЙ ОТВЕТ ИСТЦА:
{plaintiff_answer}

ИСТОРИЯ ПРОЦЕССА:
{summary_memory}

Ты — аналитический критик, задача которого выявить слабые места, логические разрывы и недостатки.

ТВОЙ АНАЛИЗ ДОЛЖЕН ВКЛЮЧАТЬ:
1. Логические ошибки и внутренние противоречия
2. Недостаточно обоснованные утверждения
3. Возможные ошибки или предположения, выданные за факты
4. Упущенные важные аспекты

КРИТИЧЕСКИЙ АНАЛИЗ (3-4 пункта):
"""
    )
    
    prompt = prompt_template.format(
        plaintiff_answer=state['plaintiff_answer'],
        summary_memory=state['summary_memory'] or "Нет истории"
    )
    
    answer = call_llm(prompt)
    state["critic_answer"] = answer
    print_agent_output("critic", answer, state['iteration'])
    
    return state

print(f"{Colors.GREEN}✓ Critic agent defined{Colors.ENDC}")

In [ ]:
def librarian_agent(state: AgentState) -> AgentState:
    """Search for sources and information"""
    prompt_template = PromptTemplate(
        input_variables=["topic"],
        template="""[ROLE: LIBRARIAN - БИБЛИОТЕКАРЬ]

ТЕМА ДЛЯ ПОИСКА:
{topic}

Ты — специалист по поиску надежной информации.

ТВОЯ ЗАДАЧА:
1. Анализируй основной вопрос/тему
2. Сформулируй поисковый запрос (2-5 слов)
3. Запрос должен быть конкретным и ориентирован на проверяемые факты

ПОИСКОВЫЙ ЗАПРОС:
"""
    )
    
    prompt = prompt_template.format(topic=state['topic'])
    query = call_llm(prompt).strip()
    
    # Perform web search
    search_results = web_search(query)
    state["librarian_sources"] = [{"query": query, "results": search_results}]
    
    print_agent_output("librarian", f"Query: {query}\n\n{search_results}", state['iteration'])
    
    return state

print(f"{Colors.GREEN}✓ Librarian agent defined{Colors.ENDC}")

In [ ]:
def juror_agent(state: AgentState, juror_id: int) -> str:
    """Individual juror opinion"""
    prompt_template = PromptTemplate(
        input_variables=["juror_id", "plaintiff_answer", "critic_answer"],
        template="""[ROLE: JUROR #{juror_id} - ПРИСЯЖНЫЙ #{juror_id}]

РАССМАТРИВАЕМЫЙ ОТВЕТ:
{plaintiff_answer}

КРИТИЧЕСКОЕ ЗАМЕЧАНИЕ:
{critic_answer}

Ты — независимый присяжный. Дай объективную оценку.

ТВОЙ АНАЛИЗ:
1. Оценка корректности (0-10)
2. Согласие или несогласие с критиком
3. Финальное суждение: приемлем ли ответ?

МНЕНИЕ ПРИСЯЖНОГО:
"""
    )
    
    prompt = prompt_template.format(
        juror_id=juror_id,
        plaintiff_answer=state['plaintiff_answer'][:300],
        critic_answer=state['critic_answer'][:200]
    )
    
    opinion = call_llm(prompt)
    print_agent_output(f"juror_{juror_id}", opinion, state['iteration'])
    
    return opinion

def jury_node(state: AgentState) -> AgentState:
    """Collect jury opinions"""
    num_jurors = 2  # Reduced for faster testing
    print(f"\n👥 Собирается жюри из {num_jurors} присяжных...\n")
    
    opinions = []
    for i in range(num_jurors):
        opinion = juror_agent(state, i + 1)
        opinions.append(opinion)
    
    state["jury_opinions"] = opinions
    state["metrics"][f"jurors_{state['iteration']}"] = num_jurors
    
    return state

print(f"{Colors.GREEN}✓ Jury agents defined{Colors.ENDC}")

In [ ]:
def judge_agent(state: AgentState) -> AgentState:
    """Judge makes final decision"""
    prompt_template = PromptTemplate(
        input_variables=["plaintiff_answer", "critic_answer", "jury_opinions"],
        template="""[ROLE: JUDGE - СУДЬЯ]

КОНТЕКСТ ДЕЛА:

ОТВЕТ ИСТЦА:
{plaintiff_answer}

КРИТИЧЕСКИЙ АНАЛИЗ:
{critic_answer}

МНЕНИЯ ПРИСЯЖНЫХ:
{jury_opinions}

Ты — финальный судья. Твоя задача — вынести обоснованное решение.

ТВОЙ ВЕРДИКТ:
1. Объективная оценка (0-10)
2. Основные достижения
3. Выявленные проблемы
4. Финальное решение: STOP или CONTINUE

РЕШЕНИЕ:
- STOP если SCORE >= 7
- CONTINUE если SCORE < 7

ВЕРДИКТ (СТРОГО В КОНЦЕ):
DECISION: [STOP|CONTINUE]
SCORE: [число 0-10]
"""
    )
    
    prompt = prompt_template.format(
        plaintiff_answer=state['plaintiff_answer'][:400],
        critic_answer=state['critic_answer'][:250],
        jury_opinions="\n---\n".join(state['jury_opinions'][:2])
    )
    
    verdict = call_llm(prompt)
    state["judge_verdict"] = verdict
    print_agent_output("judge", verdict, state['iteration'])
    
    # Parse decision
    lines = verdict.splitlines()
    decision_line = next((line for line in lines if "DECISION:" in line), "DECISION: CONTINUE")
    score_line = next((line for line in lines if "SCORE:" in line), "SCORE: 0")
    
    try:
        score = float(score_line.split("SCORE:")[-1].strip())
    except:
        score = 0.0
    
    state["stop"] = "STOP" in decision_line.upper()
    state["rewards"].append(score / 10.0 - 0.5)
    state["iteration"] += 1
    
    return state

print(f"{Colors.GREEN}✓ Judge agent defined{Colors.ENDC}")

In [ ]:
def final_agent(state: AgentState) -> AgentState:
    """Compose final answer"""
    prompt_template = PromptTemplate(
        input_variables=[
            "topic", "plaintiff_answer", "critic_answer", "jury_opinions", "judge_verdict"
        ],
        template="""[ROLE: FINALIZER - ИТОГОВЫЙ АГЕНТ]

ТЕМА: {topic}

ИМЕЮТСЯ СЛЕДУЮЩИЕ ДАННЫЕ:

ОТВЕТ ИСТЦА:
{plaintiff_answer}

КРИТИКА:
{critic_answer}

МНЕНИЯ ЖЮРИ:
{jury_opinions}

ВЕРДИКТ СУДЬИ:
{judge_verdict}

ИНСТРУКЦИЯ:
1) Синтезируй связный, окончательный ответ по теме
2) Начни с краткого резюме (1-2 предложения)
3) Основной ответ (2-4 абзаца)
4) Заключение (1-2 предложения)

ИТОГОВЫЙ ОТВЕТ:
"""
    )

    prompt = prompt_template.format(
        topic=state.get('topic', ''),
        plaintiff_answer=state.get('plaintiff_answer', '')[:400],
        critic_answer=state.get('critic_answer', '')[:250],
        jury_opinions="\n---\n".join(state.get('jury_opinions', [])),
        judge_verdict=state.get('judge_verdict', '')[:300]
    )

    answer = call_llm(prompt)
    state["final_answer"] = answer
    print_agent_output("finalizer", answer, state.get('iteration', 0))

    return state

print(f"{Colors.GREEN}✓ Final agent defined{Colors.ENDC}")

In [ ]:
def update_memory(state: AgentState) -> AgentState:
    """Update history and memory"""
    state["history"].append({
        "plaintiff": state["plaintiff_answer"][:150],
        "critic": state["critic_answer"][:100],
        "verdict": state["judge_verdict"][:80],
        "iteration": state["iteration"] - 1
    })
    
    # Create compact memory summary
    compact_history = "\n".join([
        f"Итерация {h.get('iteration', i)}: {h['plaintiff'][:60]}..."
        for i, h in enumerate(state["history"][-3:])
    ])
    
    prompt_template = PromptTemplate(
        input_variables=["history"],
        template="""Суммируй прогресс верификации в 3-4 кратких пункта.
Сосредоточься на: ошибках → исправлениях → оценке качества.

ИСТОРИЯ:
{history}

ИТОГОВАЯ СЖАТАЯ СВОДКА:
"""
    )
    
    prompt = prompt_template.format(history=compact_history)
    state["summary_memory"] = call_llm(prompt)
    
    return state

print(f"{Colors.GREEN}✓ Memory agent defined{Colors.ENDC}")

## 8. Graph Creation and Workflow

In [ ]:
def create_graph():
    """Create the agent workflow graph"""
    graph = StateGraph(AgentState)
    
    # Add nodes
    graph.add_node("plaintiff", plaintiff_agent)
    graph.add_node("critic", critic_agent)
    graph.add_node("librarian", librarian_agent)
    graph.add_node("jury", jury_node)
    graph.add_node("judge", judge_agent)
    graph.add_node("memory", update_memory)
    graph.add_node("final", final_agent)
    
    # Set entry point
    graph.set_entry_point("plaintiff")
    
    # Add edges
    graph.add_edge("plaintiff", "critic")
    graph.add_edge("critic", "librarian")
    graph.add_edge("librarian", "jury")
    graph.add_edge("jury", "judge")
    graph.add_edge("judge", "memory")
    graph.add_edge("final", END)
    
    # Conditional routing
    def should_continue(state):
        # Enforce minimum iterations
        min_iterations = 1
        if state.get("iteration", 0) < min_iterations:
            return "plaintiff"

        if state["stop"] or state["iteration"] >= state["max_iterations"]:
            return "end"

        # Early stopping if high confidence
        if state["rewards"]:
            avg_reward = sum(state["rewards"]) / len(state["rewards"])
            if avg_reward > 0.3:  # Lower threshold for notebook
                return "end"

        return "plaintiff"
    
    graph.add_conditional_edges(
        "memory",
        should_continue,
        {"plaintiff": "plaintiff", "end": "final"}
    )
    
    return graph.compile()

# Create workflow
app = create_graph()
print(f"{Colors.GREEN}✓ Workflow graph created{Colors.ENDC}")

## 9. State Initialization and Processing

In [ ]:
def initialize_state(topic: str) -> AgentState:
    """Initialize agent state"""
    return AgentState(
        topic=topic,
        plaintiff_answer="",
        critic_answer="",
        librarian_sources=[],
        jury_opinions=[],
        judge_verdict="",
        final_answer="",
        history=[],
        summary_memory="",
        iteration=0,
        max_iterations=CONFIG["max_iterations"],
        stop=False,
        metrics={},
        rewards=[],
        generated_queries=[]
    )

def process_topic(topic: str) -> Dict[str, Any]:
    """Process a single topic through the verification system"""
    print_header(f"Processing: {topic}")
    
    # Clear conversation log
    global conversation_log
    conversation_log = []
    
    state = initialize_state(topic)
    result = app.invoke(state)
    
    return {
        "topic": topic,
        "final_answer": result.get("final_answer", result.get("plaintiff_answer", "")),
        "iterations": result["iteration"],
        "judge_verdict": result["judge_verdict"],
        "rewards": result["rewards"],
    }

print(f"{Colors.GREEN}✓ Processing functions defined{Colors.ENDC}")

## 10. Test Examples

In [ ]:
# Test topics
test_topics = [
    "Что такое машинное обучение и как оно работает?",
    "Верно ли утверждение: все простые числа нечётные?"
]

print(f"{Colors.YELLOW}Test topics prepared:{Colors.ENDC}")
for i, topic in enumerate(test_topics, 1):
    print(f"  {i}. {topic}")

## 11. Run Verification System

In [ ]:
# Run system on first test topic
print(f"{Colors.BOLD}{Colors.HEADER}STARTING MULTI-AGENT VERIFICATION SYSTEM{Colors.ENDC}")
print(f"Model: QWEN-4 (Hugging Face)")
print(f"Device: {CONFIG['device']}")
print(f"Max Iterations: {CONFIG['max_iterations']}")
print()

# Process all test topics
results = []
for topic in test_topics:
    try:
        result = process_topic(topic)
        results.append(result)
    except Exception as e:
        print(f"{Colors.RED}Error processing topic: {e}{Colors.ENDC}")
        import traceback
        traceback.print_exc()
        break

## 12. Results Visualization

In [ ]:
# Display results
if results:
    print_header("FINAL VERIFICATION RESULTS")
    
    for i, result in enumerate(results, 1):
        print(f"{Colors.BOLD}{Colors.CYAN}Result {i}/{len(results)}{Colors.ENDC}")
        print(f"Topic: {result['topic']}")
        print(f"{Colors.YELLOW}Iterations: {result['iterations']}{Colors.ENDC}")
        print(f"{Colors.YELLOW}Judge Reward: {result['rewards'][-1] if result['rewards'] else 'N/A':.2f}{Colors.ENDC}")
        print(f"{Colors.GREEN}{'='*80}{Colors.ENDC}")
        print(f"\n{result['final_answer']}")
        print(f"\n{Colors.GREEN}{'='*80}{Colors.ENDC}\n")

## 13. Statistics

In [ ]:
# Print statistics
print_header("EXECUTION STATISTICS")

if conversation_log:
    total_calls = len(conversation_log)
    total_tokens = sum(item['prompt_tokens'] + item['response_tokens'] for item in conversation_log)
    avg_response_tokens = sum(item['response_tokens'] for item in conversation_log) / total_calls if total_calls > 0 else 0
    
    print(f"{Colors.YELLOW}Total LLM Calls: {total_calls}{Colors.ENDC}")
    print(f"{Colors.YELLOW}Total Tokens (estimated): {total_tokens}{Colors.ENDC}")
    print(f"{Colors.YELLOW}Average Response Tokens: {avg_response_tokens:.1f}{Colors.ENDC}")
    print()
    
    # Results summary
    print(f"{Colors.BOLD}{Colors.CYAN}Results Summary:{Colors.ENDC}")
    print(f"Topics Processed: {len(results)}")
    if results:
        avg_iterations = sum(r['iterations'] for r in results) / len(results)
        print(f"Average Iterations per Topic: {avg_iterations:.1f}")
        print()
        
        for i, result in enumerate(results, 1):
            avg_reward = sum(result['rewards']) / len(result['rewards']) if result['rewards'] else 0
            print(f"  Topic {i}: {result['topic'][:50]}... | Iterations: {result['iterations']} | Avg Reward: {avg_reward:.2f}")

## 14. Save Results to JSON

In [ ]:
# Save results to JSON
output_file = "qwen_system_results.json"

output_data = {
    "timestamp": datetime.now().isoformat(),
    "model": CONFIG["model_name"],
    "device": CONFIG["device"],
    "results": results,
    "statistics": {
        "total_llm_calls": len(conversation_log),
        "total_tokens": sum(item['prompt_tokens'] + item['response_tokens'] for item in conversation_log),
        "conversation_log": conversation_log
    }
}

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"{Colors.GREEN}✓ Results saved to: {output_file}{Colors.ENDC}")

## 15. Custom Topic Testing (Optional)

You can modify and run the cell below to test with your own topics:

In [ ]:
# Test with custom topic
custom_topic = "Какое влияние оказывает искусственный интеллект на современное общество?"

print(f"{Colors.BOLD}{Colors.CYAN}Testing custom topic...{Colors.ENDC}")
custom_result = process_topic(custom_topic)

print_header("CUSTOM TOPIC RESULT")
print(f"Topic: {custom_result['topic']}")
print(f"{Colors.YELLOW}Iterations: {custom_result['iterations']}{Colors.ENDC}")
print(f"{Colors.GREEN}{'='*80}{Colors.ENDC}")
print(custom_result["final_answer"])
print(f"{Colors.GREEN}{'='*80}{Colors.ENDC}")